# DINOv2-base phân loại lá cà chua

Chọn GPU trong phần thiết lập thời gian chạy. Tải `dinov2-source-v2.zip` và `dinov2-field-v2.zip` lên bộ nhớ phiên ở thanh Tệp trước khi giải nén. Các gói được tạo trong workspace dự án, kèm manifest và mã băm. Giữ dữ liệu ở chế độ riêng tư.

Checkpoint được chọn theo macro-F1 validation thực địa và recall của lớp không phải lá cà chua. Bước đánh giá test chỉ chạy sau khi đã chọn checkpoint. Tải weights và báo cáo về trước khi đóng runtime.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6"], check=True)
os.environ["HF_HUB_DISABLE_XET"] = "1"


In [ ]:
from zipfile import ZipFile

ROOT = Path("/content/tomato-dinov2")
ROOT.mkdir(exist_ok=True)
DATA = ROOT / "data"
RUN = ROOT / "run"
PACKAGES = {
    Path("/content/dinov2-source-v2.zip"): ROOT,
    Path("/content/dinov2-field-v2.zip"): DATA,
}
for package, target in PACKAGES.items():
    if not package.is_file():
        raise FileNotFoundError(f"Upload {package.name} to Colab session storage first")
    with ZipFile(package) as archive:
        for member in archive.infolist():
            path = target / member.filename
            if not path.resolve().is_relative_to(target.resolve()):
                raise ValueError(f"Unexpected package path: {member.filename}")
        archive.extractall(target)


In [ ]:
subprocess.run(
    [sys.executable, "-u", "-m", "ai.dinov2_field", "--data", str(DATA), "--output", str(RUN),
     "--size", "224", "--epochs", "6", "--batch", "32", "--samples", "3072", "--workers", "2"],
    cwd=ROOT, check=True,
)


In [ ]:
from zipfile import ZIP_STORED
from google.colab import files

export = Path("/content/tomato-dinov2-results.zip")
with ZipFile(export, "w", ZIP_STORED) as archive:
    for path in sorted((RUN / "model").iterdir()):
        if path.is_file():
            archive.write(path, path.relative_to(RUN))
    for name in ("history.json", "test-metrics.json", "test-predictions.csv"):
        archive.write(RUN / name, name)
print(export, export.stat().st_size)
files.download(str(export))
